# Domain Adaptation for Sentiment Analysis — Yelp Target Domain
**Source domain:** IMDB movie reviews  
**Target domain:** Yelp business/restaurant reviews  

This notebook is a direct parallel to the Amazon notebook, with Yelp as the target domain.
The purpose is to evaluate whether DAPT's contribution scales with domain distance —
Yelp reviews are expected to be more linguistically distant from movie reviews than Amazon product reviews.

## Baselines (all evaluated on Pool C — held-out Yelp test set)
| # | Baseline |
|---|----------|
| 1 | VADER (rule-based) |
| 2 | BERT + IMDB fine-tune |
| 3 | BERT + IMDB + small labelled Yelp (1000 samples) |
| 4 | BERT + IMDB + pseudo-labeling (no DAPT) |
| 5 | BERT + DAPT + IMDB fine-tune |
| 6 | BERT + DAPT + IMDB + pseudo-labeling (full pipeline) |

In [ ]:
!pip install transformers datasets evaluate scikit-learn pandas torch accelerate vaderSentiment

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 8.3 MB/s eta 0:00:00


In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import evaluate
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForMaskedLM,
    DataCollatorWithPadding,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    set_seed
)
from torch.nn.functional import softmax
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [ ]:
MODEL_NAME        = "bert-base-uncased"
NUM_LABELS        = 2
MAX_LENGTH        = 256
SEED              = 42
PSEUDO_THRESHOLD  = 0.9
SMALL_TARGET_SIZE = 1000    # labelled Yelp samples for Baseline 3
YELP_SAMPLE_SIZE  = 50000   # downsample Yelp to match Amazon dataset size

DAPT_OUTPUT_DIR         = "./yelp_bert_dapt"
IMDB_OUTPUT_DIR         = "./yelp_bert_imdb"
DAPT_IMDB_OUTPUT_DIR    = "./yelp_bert_dapt_imdb"
SMALL_TARGET_OUTPUT_DIR = "./yelp_bert_imdb_small_target"
PSEUDO_OUTPUT_DIR       = "./yelp_bert_imdb_pseudo"
PSEUDO_DAPT_OUTPUT_DIR  = "./yelp_bert_dapt_imdb_pseudo"

set_seed(SEED)

## Data Loading & Splitting

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# --- Yelp ---
# Columns: label (1=negative, 2=positive), clean_text
yelp_full_df = pd.read_csv("/content/drive/MyDrive/yelp.csv")
yelp_full_df = yelp_full_df.rename(columns={"clean_text": "text"})
yelp_full_df["label"] = yelp_full_df["label"].map({1: 0, 2: 1})  # 1 -> 0 (neg), 2 -> 1 (pos)
yelp_full_df = yelp_full_df[["text", "label"]].dropna()

# Downsample to 50k to match Amazon dataset size (stratified to preserve balance)
yelp_df, _ = train_test_split(
    yelp_full_df, train_size=YELP_SAMPLE_SIZE,
    stratify=yelp_full_df["label"], random_state=SEED
)
yelp_df = yelp_df.reset_index(drop=True)

# --- IMDB ---
imdb_df = pd.read_csv("/content/drive/MyDrive/IMDB.csv")
imdb_df = imdb_df.rename(columns={"review": "text", "sentiment": "label"})
imdb_df["label"] = imdb_df["label"].map({"negative": 0, "positive": 1})
imdb_df = imdb_df[["text", "label"]].dropna()

print(f"Yelp (downsampled): {len(yelp_df)} samples | label dist: {yelp_df['label'].value_counts().to_dict()}")
print(f"IMDB:               {len(imdb_df)} samples | label dist: {imdb_df['label'].value_counts().to_dict()}")

Mounted at /content/drive
Yelp (downsampled): 50000 samples | label dist: {1: 25004, 0: 24996}
IMDB:               50000 samples | label dist: {1: 25000, 0: 25000}


In [ ]:
# =========================
# Yelp pool splitting (60 / 20 / 20, stratified)
# =========================

# Pool A (60%) — raw text for DAPT, labels stripped
pool_a_df, pool_bc_df = train_test_split(
    yelp_df, test_size=0.4,
    stratify=yelp_df["label"], random_state=SEED
)

# Pool B (20%) — unlabelled, for pseudo-labeling
# Pool C (20%) — held-out test set
pool_b_df, pool_c_df = train_test_split(
    pool_bc_df, test_size=0.5,
    stratify=pool_bc_df["label"], random_state=SEED
)

# Strip labels from Pool A (DAPT uses no label information)
pool_a_text_df = pool_a_df[["text"]].reset_index(drop=True)

# Strip labels from Pool B (pseudo-labeling starts from unlabelled text)
pool_b_unlabeled_df = pool_b_df[["text"]].reset_index(drop=True)

# Small labelled target set for Baseline 3 (sampled from Pool B, stratified)
small_target_df, _ = train_test_split(
    pool_b_df, train_size=SMALL_TARGET_SIZE,
    stratify=pool_b_df["label"], random_state=SEED
)

print(f"Pool A (DAPT):            {len(pool_a_df):>6} samples (labels stripped)")
print(f"Pool B (pseudo-labeling): {len(pool_b_df):>6} samples (unlabelled)")
print(f"Pool C (held-out test):   {len(pool_c_df):>6} samples")
print(f"Small labelled target:    {len(small_target_df):>6} samples (from Pool B)")

Pool A (DAPT):             30000 samples (labels stripped)
Pool B (pseudo-labeling):  10000 samples (unlabelled)
Pool C (held-out test):    10000 samples
Small labelled target:      1000 samples (from Pool B)


In [ ]:
# =========================
# IMDB train/validation split
# =========================
imdb_train_df, imdb_valid_df = train_test_split(
    imdb_df, test_size=0.2,
    stratify=imdb_df["label"], random_state=SEED
)

print(f"IMDB train: {len(imdb_train_df)} | IMDB valid: {len(imdb_valid_df)}")

IMDB train: 40000 | IMDB valid: 10000


## Tokenizer & Shared Utilities

In [ ]:
# =========================
# Tokenizer
# =========================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_clf(examples):
    """Tokenizer for classification tasks (has labels)."""
    return tokenizer(examples["text"], truncation=True, max_length=MAX_LENGTH)

def tokenize_mlm(examples):
    """Tokenizer for MLM / DAPT (text only, no labels)."""
    return tokenizer(examples["text"], truncation=True, max_length=MAX_LENGTH)

data_collator_clf = DataCollatorWithPadding(tokenizer=tokenizer)
data_collator_mlm = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
# =========================
# Metrics
# =========================
accuracy_metric = evaluate.load("accuracy")
f1_metric       = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1  = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    precision, recall, f1_per_class, _ = precision_recall_fscore_support(
        labels, preds, average=None, zero_division=0
    )

    metrics = {"accuracy": acc, "macro_f1": f1}
    for i in range(len(precision)):
        metrics[f"precision_class_{i}"] = precision[i]
        metrics[f"recall_class_{i}"]    = recall[i]
        metrics[f"f1_class_{i}"]        = f1_per_class[i]
    return metrics


def build_training_args(output_dir, lr=2e-5, epochs=3, train_batch=16, eval_batch=32):
    """Shared TrainingArguments factory for classification fine-tuning."""
    return TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        learning_rate=lr,
        per_device_train_batch_size=train_batch,
        per_device_eval_batch_size=eval_batch,
        num_train_epochs=epochs,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=2,
        report_to="none",
        fp16=torch.cuda.is_available(),
        seed=SEED
    )


def evaluate_on_pool_c(trainer, pool_c_tok, label):
    """Evaluate a trainer on Pool C and return a results dict."""
    print(f"\n--- Evaluating: {label} ---")
    results = trainer.evaluate(eval_dataset=pool_c_tok)
    print(results)
    return {"baseline": label, **results}

In [ ]:
# =========================
# Tokenize shared datasets
# =========================

# IMDB
imdb_train_tok = Dataset.from_pandas(imdb_train_df.reset_index(drop=True)).map(tokenize_clf, batched=True)
imdb_valid_tok = Dataset.from_pandas(imdb_valid_df.reset_index(drop=True)).map(tokenize_clf, batched=True)

# Pool C (held-out Yelp test set)
pool_c_tok = Dataset.from_pandas(pool_c_df.reset_index(drop=True)).map(tokenize_clf, batched=True)

# Pool A (DAPT — MLM, text only)
pool_a_tok = Dataset.from_pandas(pool_a_text_df).map(tokenize_mlm, batched=True)

# Pool B (unlabelled — for pseudo-labeling inference)
pool_b_tok = Dataset.from_pandas(pool_b_unlabeled_df).map(tokenize_mlm, batched=True)

# Small labelled Yelp (Baseline 3)
small_target_tok = Dataset.from_pandas(small_target_df.reset_index(drop=True)).map(tokenize_clf, batched=True)

Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/30000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

## Baseline 1 — VADER (Rule-Based)

In [ ]:
# =========================
# Baseline 1: VADER
# =========================
analyzer = SentimentIntensityAnalyzer()

def vader_predict(texts, threshold=0.05):
    """
    Compound score >= threshold -> positive (1)
    Compound score <  threshold -> negative (0)
    Threshold of 0.05 is the standard VADER recommendation.
    """
    preds = []
    for text in texts:
        score = analyzer.polarity_scores(str(text))["compound"]
        preds.append(1 if score >= threshold else 0)
    return np.array(preds)

vader_preds  = vader_predict(pool_c_df["text"].tolist())
vader_labels = pool_c_df["label"].values

vader_acc = accuracy_score(vader_labels, vader_preds)
vader_f1  = f1_score(vader_labels, vader_preds, average="macro")

print(f"VADER | Accuracy: {vader_acc:.4f} | Macro F1: {vader_f1:.4f}")

baseline_results = [
    {"baseline": "1. VADER", "eval_accuracy": vader_acc, "eval_macro_f1": vader_f1}
]

VADER | Accuracy: 0.7199 | Macro F1: 0.7015


## Baseline 2 — BERT + IMDB Fine-tune

In [ ]:
# =========================
# Baseline 2: BERT + IMDB
# =========================
imdb_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)

imdb_trainer = Trainer(
    model=imdb_model,
    args=build_training_args(IMDB_OUTPUT_DIR, lr=2e-5, epochs=3),
    train_dataset=imdb_train_tok,
    eval_dataset=imdb_valid_tok,
    data_collator=data_collator_clf,
    compute_metrics=compute_metrics
)

print("===== Baseline 2: BERT + IMDB fine-tune =====")
imdb_trainer.train()
imdb_trainer.save_model(IMDB_OUTPUT_DIR)
tokenizer.save_pretrained(IMDB_OUTPUT_DIR)

baseline_results.append(evaluate_on_pool_c(imdb_trainer, pool_c_tok, "2. BERT + IMDB"))

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


===== Baseline 2: BERT + IMDB fine-tune =====


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Precision Class 0,Recall Class 0,F1 Class 0,Precision Class 1,Recall Class 1,F1 Class 1
1,0.259850,0.214765,0.923600,0.923593,0.931892,0.914000,0.922859,0.915620,0.933200,0.924326
2,0.152536,0.236344,0.926400,0.926377,0.911742,0.944200,0.927687,0.942140,0.908600,0.925066
3,0.083307,0.309854,0.929300,0.929296,0.935484,0.922200,0.928794,0.923289,0.936400,0.929798


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Evaluating: 2. BERT + IMDB ---


{'eval_loss': 0.3134591281414032, 'eval_accuracy': 0.9038, 'eval_macro_f1': 0.9037369130233189, 'eval_precision_class_0': 0.925395152792413, 'eval_recall_class_0': 0.878375675135027, 'eval_f1_class_0': 0.9012725779967159, 'eval_precision_class_1': 0.8843006660323501, 'eval_recall_class_1': 0.9292141571685663, 'eval_f1_class_1': 0.906201248049922, 'eval_runtime': 44.1803, 'eval_samples_per_second': 226.345, 'eval_steps_per_second': 7.085, 'epoch': 3.0}


## Baseline 3 — BERT + IMDB + Small Labelled Yelp (1000 samples)

In [ ]:
# =========================
# Baseline 3: BERT + IMDB + small labelled Yelp
# Start from the saved IMDB model, fine-tune further on 1000 labelled Yelp samples
# =========================
small_target_model = AutoModelForSequenceClassification.from_pretrained(IMDB_OUTPUT_DIR, num_labels=NUM_LABELS)

small_target_trainer = Trainer(
    model=small_target_model,
    args=build_training_args(SMALL_TARGET_OUTPUT_DIR, lr=1e-5, epochs=5, train_batch=8),
    train_dataset=small_target_tok,
    eval_dataset=pool_c_tok,
    data_collator=data_collator_clf,
    compute_metrics=compute_metrics
)

print("===== Baseline 3: BERT + IMDB + small labelled Yelp =====")
small_target_trainer.train()
small_target_trainer.save_model(SMALL_TARGET_OUTPUT_DIR)

baseline_results.append(evaluate_on_pool_c(small_target_trainer, pool_c_tok, "3. BERT + IMDB + Small Labelled Yelp"))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

===== Baseline 3: BERT + IMDB + small labelled Yelp =====


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Precision Class 0,Recall Class 0,F1 Class 0,Precision Class 1,Recall Class 1,F1 Class 1
1,0.317735,0.246435,0.936400,0.936400,0.938492,0.933987,0.936234,0.934328,0.938812,0.936565
2,0.140605,0.301880,0.933000,0.932975,0.916651,0.952591,0.934275,0.950676,0.913417,0.931674
3,0.067161,0.334516,0.934600,0.934598,0.939333,0.929186,0.934232,0.929970,0.940012,0.934964
4,0.032646,0.349545,0.934100,0.934094,0.942676,0.924385,0.933441,0.925853,0.943811,0.934746
5,0.017011,0.353575,0.935100,0.935100,0.933440,0.936987,0.935210,0.936772,0.933213,0.934989


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Evaluating: 3. BERT + IMDB + Small Labelled Yelp ---


{'eval_loss': 0.24696142971515656, 'eval_accuracy': 0.9363, 'eval_macro_f1': 0.9362996018725116, 'eval_precision_class_0': 0.9383038585209004, 'eval_recall_class_0': 0.9339867973594719, 'eval_f1_class_0': 0.9361403508771929, 'eval_precision_class_1': 0.9343152866242038, 'eval_recall_class_1': 0.9386122775444911, 'eval_f1_class_1': 0.9364588528678304, 'eval_runtime': 38.4993, 'eval_samples_per_second': 259.745, 'eval_steps_per_second': 8.13, 'epoch': 5.0}


## Baseline 4 — BERT + IMDB + Pseudo-labeling (No DAPT)

In [ ]:
# =========================
# Pseudo-label generation utility
# =========================
def generate_pseudo_labels(trainer, unlabeled_tok, threshold=PSEUDO_THRESHOLD):
    """
    Run inference on an unlabelled tokenized dataset.
    Keep only samples where max softmax probability >= threshold.
    Returns a tokenized Dataset with pseudo labels attached.
    """
    outputs           = trainer.predict(unlabeled_tok)
    probs             = softmax(torch.tensor(outputs.predictions), dim=1)
    confidences, preds = torch.max(probs, dim=1)
    mask              = confidences >= threshold

    kept_texts  = [unlabeled_tok[i]["text"] for i in range(len(unlabeled_tok)) if mask[i]]
    kept_labels = preds[mask].numpy()

    print(f"Pseudo-labeled samples kept: {len(kept_texts)} / {len(unlabeled_tok)} "
          f"(threshold={threshold})")

    pseudo_df  = pd.DataFrame({"text": kept_texts, "label": kept_labels})
    pseudo_ds  = Dataset.from_pandas(pseudo_df.reset_index(drop=True))
    pseudo_tok = pseudo_ds.map(tokenize_clf, batched=True)
    return pseudo_tok

In [ ]:
# =========================
# Baseline 4: BERT + IMDB + pseudo-labeling (no DAPT)
# Teacher: IMDB-only model (Baseline 2)
# =========================
pseudo_b4_tok = generate_pseudo_labels(imdb_trainer, pool_b_tok)

pseudo_b4_model = AutoModelForSequenceClassification.from_pretrained(IMDB_OUTPUT_DIR, num_labels=NUM_LABELS)

pseudo_b4_trainer = Trainer(
    model=pseudo_b4_model,
    args=build_training_args(PSEUDO_OUTPUT_DIR, lr=1e-5, epochs=3, train_batch=8),
    train_dataset=pseudo_b4_tok,
    eval_dataset=pool_c_tok,
    data_collator=data_collator_clf,
    compute_metrics=compute_metrics
)

print("===== Baseline 4: BERT + IMDB + Pseudo-labeling (no DAPT) =====")
pseudo_b4_trainer.train()
pseudo_b4_trainer.save_model(PSEUDO_OUTPUT_DIR)

baseline_results.append(evaluate_on_pool_c(pseudo_b4_trainer, pool_c_tok, "4. BERT + IMDB + Pseudo (no DAPT)"))

Pseudo-labeled samples kept: 9088 / 10000 (threshold=0.9)


Map:   0%|          | 0/9088 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

===== Baseline 4: BERT + IMDB + Pseudo-labeling (no DAPT) =====


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Precision Class 0,Recall Class 0,F1 Class 0,Precision Class 1,Recall Class 1,F1 Class 1
1,0.120361,0.551725,0.905400,0.905205,0.945678,0.860172,0.900901,0.871814,0.950610,0.909508
2,0.042706,0.621136,0.916000,0.915941,0.939176,0.889578,0.913705,0.895157,0.942412,0.918177
3,0.013487,0.647240,0.916100,0.916048,0.937710,0.891378,0.913958,0.896532,0.940812,0.918138


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Evaluating: 4. BERT + IMDB + Pseudo (no DAPT) ---


{'eval_loss': 0.647240400314331, 'eval_accuracy': 0.9161, 'eval_macro_f1': 0.9160479488887905, 'eval_precision_class_0': 0.9377104377104377, 'eval_recall_class_0': 0.8913782756551311, 'eval_f1_class_0': 0.9139575428161214, 'eval_precision_class_1': 0.8965320121951219, 'eval_recall_class_1': 0.9408118376324736, 'eval_f1_class_1': 0.9181383549614597, 'eval_runtime': 44.4495, 'eval_samples_per_second': 224.974, 'eval_steps_per_second': 7.042, 'epoch': 3.0}


## Baseline 5 — BERT + DAPT + IMDB Fine-tune

In [ ]:
# =========================
# Baseline 5 (Step 1): DAPT — continue MLM pretraining on Pool A (Yelp text)
# =========================
dapt_model = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)

dapt_training_args = TrainingArguments(
    output_dir=DAPT_OUTPUT_DIR,
    eval_strategy="no",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=1,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED
)

dapt_trainer = Trainer(
    model=dapt_model,
    args=dapt_training_args,
    train_dataset=pool_a_tok,
    data_collator=data_collator_mlm,
)

print("===== DAPT: Continued MLM pretraining on Pool A (Yelp) =====")
dapt_trainer.train()
dapt_trainer.save_model(DAPT_OUTPUT_DIR)
tokenizer.save_pretrained(DAPT_OUTPUT_DIR)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


===== DAPT: Continued MLM pretraining on Pool A (Yelp) =====


Step,Training Loss
1875,2.829281
3750,2.520171
5625,2.365865


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./yelp_bert_dapt/tokenizer_config.json', './yelp_bert_dapt/tokenizer.json')

In [ ]:
# =========================
# Baseline 5 (Step 2): Fine-tune DAPT model on IMDB
# =========================
dapt_imdb_model = AutoModelForSequenceClassification.from_pretrained(DAPT_OUTPUT_DIR, num_labels=NUM_LABELS)

dapt_imdb_trainer = Trainer(
    model=dapt_imdb_model,
    args=build_training_args(DAPT_IMDB_OUTPUT_DIR, lr=2e-5, epochs=3),
    train_dataset=imdb_train_tok,
    eval_dataset=imdb_valid_tok,
    data_collator=data_collator_clf,
    compute_metrics=compute_metrics
)

print("===== Baseline 5: BERT + DAPT + IMDB fine-tune =====")
dapt_imdb_trainer.train()
dapt_imdb_trainer.save_model(DAPT_IMDB_OUTPUT_DIR)
tokenizer.save_pretrained(DAPT_IMDB_OUTPUT_DIR)

baseline_results.append(evaluate_on_pool_c(dapt_imdb_trainer, pool_c_tok, "5. BERT + DAPT + IMDB"))

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ./yelp_bert_dapt
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


===== Baseline 5: BERT + DAPT + IMDB fine-tune =====


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Precision Class 0,Recall Class 0,F1 Class 0,Precision Class 1,Recall Class 1,F1 Class 1
1,0.252198,0.223568,0.921700,0.921647,0.944925,0.895600,0.919602,0.900779,0.947800,0.923692
2,0.146702,0.257058,0.924200,0.924194,0.917027,0.932800,0.924846,0.931624,0.915600,0.923542
3,0.077927,0.319589,0.928500,0.928495,0.936088,0.919800,0.927872,0.921172,0.937200,0.929117


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Evaluating: 5. BERT + DAPT + IMDB ---


{'eval_loss': 0.23156410455703735, 'eval_accuracy': 0.9279, 'eval_macro_f1': 0.9278363451617037, 'eval_precision_class_0': 0.954719387755102, 'eval_recall_class_0': 0.8983796759351871, 'eval_f1_class_0': 0.9256930846130063, 'eval_precision_class_1': 0.9040785498489426, 'eval_recall_class_1': 0.9574085182963408, 'eval_f1_class_1': 0.9299796057104011, 'eval_runtime': 44.4683, 'eval_samples_per_second': 224.879, 'eval_steps_per_second': 7.039, 'epoch': 3.0}


## Baseline 6 — Full Pipeline: BERT + DAPT + IMDB + Pseudo-labeling

In [ ]:
# =========================
# Baseline 6: Pseudo-labeling using DAPT+IMDB teacher
# =========================
pseudo_b6_tok = generate_pseudo_labels(dapt_imdb_trainer, pool_b_tok)

pseudo_b6_model = AutoModelForSequenceClassification.from_pretrained(DAPT_IMDB_OUTPUT_DIR, num_labels=NUM_LABELS)

pseudo_b6_trainer = Trainer(
    model=pseudo_b6_model,
    args=build_training_args(PSEUDO_DAPT_OUTPUT_DIR, lr=1e-5, epochs=3, train_batch=8),
    train_dataset=pseudo_b6_tok,
    eval_dataset=pool_c_tok,
    data_collator=data_collator_clf,
    compute_metrics=compute_metrics
)

print("===== Baseline 6: BERT + DAPT + IMDB + Pseudo-labeling (Full Pipeline) =====")
pseudo_b6_trainer.train()
pseudo_b6_trainer.save_model(PSEUDO_DAPT_OUTPUT_DIR)

baseline_results.append(evaluate_on_pool_c(pseudo_b6_trainer, pool_c_tok, "6. BERT + DAPT + IMDB + Pseudo (Full)"))

Pseudo-labeled samples kept: 9222 / 10000 (threshold=0.9)


Map:   0%|          | 0/9222 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

===== Baseline 6: BERT + DAPT + IMDB + Pseudo-labeling (Full Pipeline) =====


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Precision Class 0,Recall Class 0,F1 Class 0,Precision Class 1,Recall Class 1,F1 Class 1
1,0.066920,0.456731,0.934700,0.934665,0.955556,0.911782,0.933156,0.915679,0.957608,0.936174
2,0.020929,0.504172,0.935600,0.935563,0.957362,0.911782,0.934016,0.915824,0.959408,0.937109
3,0.004837,0.530096,0.936500,0.936455,0.960921,0.909982,0.934758,0.914546,0.963007,0.938151


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


--- Evaluating: 6. BERT + DAPT + IMDB + Pseudo (Full) ---


{'eval_loss': 0.5300956964492798, 'eval_accuracy': 0.9365, 'eval_macro_f1': 0.936454699190506, 'eval_precision_class_0': 0.9609209970426701, 'eval_recall_class_0': 0.9099819963992799, 'eval_f1_class_0': 0.9347580396588925, 'eval_precision_class_1': 0.9145461450816559, 'eval_recall_class_1': 0.9630073985202959, 'eval_f1_class_1': 0.9381513587221194, 'eval_runtime': 44.579, 'eval_samples_per_second': 224.321, 'eval_steps_per_second': 7.021, 'epoch': 3.0}


## Final Comparison Table

In [ ]:
# =========================
# Final comparison table
# =========================
summary = pd.DataFrame([
    {
        "Baseline": r["baseline"],
        "Accuracy": round(r["eval_accuracy"], 4),
        "Macro F1": round(r["eval_macro_f1"], 4),
    }
    for r in baseline_results
])

print("\n===== Final Results on Pool C (held-out Yelp test set) =====")
print(summary.to_string(index=False))


===== Final Results on Pool C (held-out Yelp test set) =====
                             Baseline  Accuracy  Macro F1
                             1. VADER    0.7199    0.7015
                       2. BERT + IMDB    0.9038    0.9037
 3. BERT + IMDB + Small Labelled Yelp    0.9363    0.9363
    4. BERT + IMDB + Pseudo (no DAPT)    0.9161    0.9160
                5. BERT + DAPT + IMDB    0.9279    0.9278
6. BERT + DAPT + IMDB + Pseudo (Full)    0.9365    0.9365
